In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 30


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.769188791513443
Epoch 2/100, Loss: 3.014891415834427
Epoch 3/100, Loss: 2.8860966935753822
Epoch 4/100, Loss: 2.755319744348526
Epoch 5/100, Loss: 2.832272842526436
Epoch 6/100, Loss: 3.024959720671177
Epoch 7/100, Loss: 2.828670673072338
Epoch 8/100, Loss: 2.8460957184433937
Epoch 9/100, Loss: 2.9991803094744682
Epoch 10/100, Loss: 2.98090523481369
Epoch 11/100, Loss: 3.053616523742676
Epoch 12/100, Loss: 2.927617385983467
Epoch 13/100, Loss: 3.008375607430935
Epoch 14/100, Loss: 3.016100436449051
Epoch 15/100, Loss: 2.8698650747537613
Epoch 16/100, Loss: 2.9904593154788017
Epoch 17/100, Loss: 2.756232500076294
Epoch 18/100, Loss: 2.940292038023472


Epoch 19/100, Loss: 2.93275585770607
Epoch 20/100, Loss: 2.831635534763336
Epoch 21/100, Loss: 2.9085763469338417
Epoch 22/100, Loss: 3.0848161727190018
Epoch 23/100, Loss: 3.174006626009941
Epoch 24/100, Loss: 2.988541804254055
Epoch 25/100, Loss: 2.8845801949501038
Epoch 26/100, Loss: 2.9749146699905396
Epoch 27/100, Loss: 2.9797525629401207
Epoch 28/100, Loss: 2.807102032005787
Epoch 29/100, Loss: 2.9989799708127975
Epoch 30/100, Loss: 2.746422067284584
Epoch 31/100, Loss: 3.0022700801491737
Epoch 32/100, Loss: 2.9091861099004745
Epoch 33/100, Loss: 3.174999713897705
Epoch 34/100, Loss: 2.994252599775791
Epoch 35/100, Loss: 2.8916156589984894
Epoch 36/100, Loss: 2.859381079673767


Epoch 37/100, Loss: 2.9699730798602104
Epoch 38/100, Loss: 2.728495642542839
Epoch 39/100, Loss: 2.916208043694496
Epoch 40/100, Loss: 3.131091170012951
Epoch 41/100, Loss: 2.7580994218587875
Epoch 42/100, Loss: 3.121324971318245
Epoch 43/100, Loss: 2.5638851523399353
Epoch 44/100, Loss: 2.8274944946169853
Epoch 45/100, Loss: 3.1031890138983727
Epoch 46/100, Loss: 2.9843365773558617
Epoch 47/100, Loss: 3.0618879720568657
Epoch 48/100, Loss: 2.880380965769291
Epoch 49/100, Loss: 2.986400991678238
Epoch 50/100, Loss: 3.0251591876149178
Epoch 51/100, Loss: 2.8706905767321587
Epoch 52/100, Loss: 2.7443622574210167
Epoch 53/100, Loss: 2.9847329407930374


Epoch 54/100, Loss: 2.8950291872024536
Epoch 55/100, Loss: 2.891917183995247
Epoch 56/100, Loss: 2.86015035957098
Epoch 57/100, Loss: 3.118316560983658
Epoch 58/100, Loss: 3.0475757345557213
Epoch 59/100, Loss: 3.033180020749569
Epoch 60/100, Loss: 2.723597154021263
Epoch 61/100, Loss: 3.062032863497734
Epoch 62/100, Loss: 2.895999602973461
Epoch 63/100, Loss: 2.9944416284561157
Epoch 64/100, Loss: 3.153621904551983
Epoch 65/100, Loss: 2.9584123864769936
Epoch 66/100, Loss: 2.9973212257027626
Epoch 67/100, Loss: 3.0149300768971443
Epoch 68/100, Loss: 2.841122455894947
Epoch 69/100, Loss: 2.9150127544999123
Epoch 70/100, Loss: 3.6788924857974052
Epoch 71/100, Loss: 2.738530106842518


Epoch 72/100, Loss: 2.7992780655622482
Epoch 73/100, Loss: 2.908010244369507
Epoch 74/100, Loss: 3.153112292289734
Epoch 75/100, Loss: 3.312899298965931
Epoch 76/100, Loss: 2.999241054058075
Epoch 77/100, Loss: 3.3163692206144333
Epoch 78/100, Loss: 3.1564828976988792
Epoch 79/100, Loss: 3.0221060439944267
Epoch 80/100, Loss: 2.832527406513691
Epoch 81/100, Loss: 3.0044369846582413
Epoch 82/100, Loss: 2.831817165017128
Epoch 83/100, Loss: 3.0430444180965424
Epoch 84/100, Loss: 3.0318364799022675
Epoch 85/100, Loss: 2.801310256123543
Epoch 86/100, Loss: 2.937437027692795
Epoch 87/100, Loss: 3.045528121292591
Epoch 88/100, Loss: 3.056598499417305


...

Epoch 9/100, Loss: 2.698346309363842
Epoch 10/100, Loss: 2.677822567522526
Epoch 11/100, Loss: 2.711311899125576
Epoch 12/100, Loss: 2.5451513826847076
Epoch 13/100, Loss: 2.629963167011738
Epoch 14/100, Loss: 2.634024292230606
Epoch 15/100, Loss: 2.6271577179431915
Epoch 16/100, Loss: 2.668494962155819
Epoch 17/100, Loss: 2.5359936952590942
Epoch 18/100, Loss: 2.7580240666866302
Epoch 19/100, Loss: 2.671355254948139
Epoch 20/100, Loss: 2.6221652030944824
Epoch 21/100, Loss: 2.724247097969055
Epoch 22/100, Loss: 2.5686386302113533
Epoch 23/100, Loss: 2.6583022996783257
Epoch 24/100, Loss: 2.5293226167559624
Epoch 25/100, Loss: 2.4927446395158768
Epoch 26/100, Loss: 2.7277810350060463


Epoch 27/100, Loss: 2.607497327029705
Epoch 28/100, Loss: 2.500982142984867
Epoch 29/100, Loss: 2.6571793779730797
Epoch 30/100, Loss: 2.6981864273548126
Epoch 31/100, Loss: 2.486561819911003
Epoch 32/100, Loss: 2.5135063603520393
Epoch 33/100, Loss: 2.7108307257294655
Epoch 34/100, Loss: 2.9021814316511154
Epoch 35/100, Loss: 2.6017519682645798
Epoch 36/100, Loss: 2.5485417023301125
Epoch 37/100, Loss: 2.7165661081671715
Epoch 38/100, Loss: 2.6915832310914993
Epoch 39/100, Loss: 2.75791098177433
Epoch 40/100, Loss: 2.5835047215223312
Epoch 41/100, Loss: 2.56636880338192
Epoch 42/100, Loss: 2.729457475244999
Epoch 43/100, Loss: 2.745676651597023
Epoch 44/100, Loss: 2.7158216163516045


Epoch 45/100, Loss: 2.7369758039712906
Epoch 46/100, Loss: 2.668954536318779
Epoch 47/100, Loss: 2.728962540626526
Epoch 48/100, Loss: 2.7067083343863487
Epoch 49/100, Loss: 2.6695386096835136
Epoch 50/100, Loss: 2.6294176802039146
Epoch 51/100, Loss: 2.736249700188637
Epoch 52/100, Loss: 2.609336346387863
Epoch 53/100, Loss: 2.8909915685653687
Epoch 54/100, Loss: 2.5825091376900673
Epoch 55/100, Loss: 2.7773258686065674
Epoch 56/100, Loss: 2.588426873087883
Epoch 57/100, Loss: 2.747415781021118
Epoch 58/100, Loss: 2.724720686674118
Epoch 59/100, Loss: 2.706505388021469
Epoch 60/100, Loss: 2.733992248773575
Epoch 61/100, Loss: 2.599690616130829
Epoch 62/100, Loss: 2.650760993361473
Epoch 63/100, Loss: 2.5649536550045013


Epoch 64/100, Loss: 2.661088742315769
Epoch 65/100, Loss: 2.5765172243118286
Epoch 66/100, Loss: 2.740207351744175
Epoch 67/100, Loss: 2.625429928302765
Epoch 68/100, Loss: 2.7044700160622597
Epoch 69/100, Loss: 2.6590830385684967
Epoch 70/100, Loss: 2.4713737070560455
Epoch 71/100, Loss: 2.5130219012498856
Epoch 72/100, Loss: 2.705066978931427
Epoch 73/100, Loss: 2.4954039976000786
Epoch 74/100, Loss: 2.750152438879013
Epoch 75/100, Loss: 2.729416109621525
Epoch 76/100, Loss: 2.6841071471571922
Epoch 77/100, Loss: 2.5799440294504166
Epoch 78/100, Loss: 2.708591528236866
Epoch 79/100, Loss: 2.597497507929802
Epoch 80/100, Loss: 2.682851828634739
Epoch 81/100, Loss: 2.6803917959332466
Epoch 82/100, Loss: 2.8176154494285583
Epoch 83/100, Loss: 2.716831237077713


Epoch 84/100, Loss: 2.5347402840852737
Epoch 85/100, Loss: 2.5742284953594208
Epoch 86/100, Loss: 2.6454270407557487
Epoch 87/100, Loss: 2.612947002053261
Epoch 88/100, Loss: 2.487403944134712
Epoch 89/100, Loss: 2.6121410205960274
Epoch 90/100, Loss: 2.6738604083657265
Epoch 91/100, Loss: 2.7799170315265656
Epoch 92/100, Loss: 2.720024362206459
Epoch 93/100, Loss: 2.793080873787403
Epoch 94/100, Loss: 2.695739060640335
Epoch 95/100, Loss: 2.502223029732704
Epoch 96/100, Loss: 2.6812151223421097
Epoch 97/100, Loss: 2.801949977874756
Epoch 98/100, Loss: 2.520612895488739
Epoch 99/100, Loss: 2.5957290902733803
Epoch 100/100, Loss: 2.61372809112072
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.6327
